## Exercise 2

In [ ]:
# part 1
# Choose a value for the probability parameter (p) in the geometric distribution and
#simulate 10,000 observations.

#Compare the simulated and theoretical distributions. You should experiment with
#small, moderate, and large values of p.

import numpy as np
import RNG_mat
from scipy import stats
import matplotlib.pyplot as plt
sample_size = 10_000
np.random.seed(67)

p = 0.1

def geometric_dist(p,sample_size, plot_=True):
    geometric_numbers = np.random.geometric(p=p, size=sample_size)
    n, count = np.unique(geometric_numbers, return_counts=True)
    theoretical_geometric_numbers = p*(1-p)**(n-1)
    if plot_:
        plt.hist(geometric_numbers, color='r', bins=np.arange(1,np.max(geometric_numbers))-1/2, density=True)
        plt.plot(n,theoretical_geometric_numbers, color='b')
    return geometric_numbers, theoretical_geometric_numbers

geometric_numbers, theoretical_geometric_numbers = geometric_dist(p, sample_size)


In [ ]:
# part 2
# with a 6-point distribution 
np.random.seed(67)




x = np.array([1,2,3,4,5,6])
P = np.array([7/48, 5/48, 1/8, 1/16, 1/4, 5/16])
F = np.cumsum(P)

# generate observatios from this distribution using 
# 1. the direct cude method

def crude_dist(P, x,sample_size, plot_=True):
    U = np.random.uniform(0,1, sample_size)
    crude_x = []
    for u in U:
        if 0<u and u<F[0]:
            crude_x.append(int(x[0]))
        else:
            for i in range(1,len(x)):
                if F[i-1]<u and u<F[i]:
                    crude_x.append(int(x[i]))
                    break
    n, counts = np.unique(crude_x, return_counts=True)
    mean_counts = counts/np.sum(counts)
    if plot_:
        plt.bar(x+0.2,P,color='b' , width=0.4,label = 'Theoretical')
        plt.bar(n-0.2 ,mean_counts, color='r', width=0.4, label='simulated crude')
        plt.title('Crude distributions')
    return mean_counts


crude_dist_points = crude_dist(P,x,sample_size)



In [ ]:
# 2. the rejection method
np.random.seed(67)
x = np.array([1,2,3,4,5,6])
P = np.array([7/48, 5/48, 1/8, 1/16, 1/4, 5/16])


def rejection_method(P,x,sample_size,plot_=True):
    k = x[-1]
    c = np.max(P)+0.05

    simple_reject_x = []
    while len(simple_reject_x)<sample_size:
        U1 = np.random.rand()
        U2 = np.random.rand()
        I = int(np.floor(k*U1 ))

        if U2<= P[I]/c:
            simple_reject_x.append(x[I])




    n, counts = np.unique(simple_reject_x, return_counts=True)
    mean_counts = counts/np.sum(counts)

    if plot_:
        plt.bar(x+0.2,P,color='b' , width=0.4,label = 'Theoretical')
        plt.bar(n-0.2 ,mean_counts, color='r', width=0.4, label='simple rejection')
        plt.title('simple rejection distributions')
    return mean_counts

rejection_points = rejection_method(P,x,sample_size)

In [ ]:
# 3. the alias method 

np.random.seed(67)
x = np.array([1,2,3,4,5,6])
P = np.array([7/48, 5/48, 1/8, 1/16, 1/4, 5/16])

def alias(P,x,sample_size,plot_=True):
    k = x[-1]

    # Alias-method setup
    F = k * P
    prob = F.copy()
    alias = np.zeros(k, dtype=int)

    small = [i for i in range(k) if prob[i] < 1]
    large = [i for i in range(k) if prob[i] >= 1]

    while small and large:
        s = small.pop()
        l = large.pop()
        alias[s] = l
        prob[l] = prob[l] + prob[s] - 1.0
        if prob[l] < 1:
            small.append(l)
        else:
            large.append(l)

    # Make sure every entry is a valid probability in [0, 1]
    for i in small:
        prob[i] = 1.0
    for i in large:
        prob[i] = 1.0

    # Generate samples with the alias table
    alias_samples = np.empty(sample_size, dtype=int)
    for i in range(sample_size):
        u1 = np.random.rand()
        u2 = np.random.rand()
        j = int(np.floor(k * u1))
        alias_samples[i] = x[j] if u2 < prob[j] else x[alias[j]]


    n_alias, counts_alias = np.unique(alias_samples, return_counts=True)
    mean_alias = counts_alias / np.sum(counts_alias)

    if plot_:
        plt.bar(x + 0.2, P, color='b', width=0.4, label='Theoretical')
        plt.bar(n_alias - 0.2, mean_alias, color='r', width=0.4, label='Alias method')
        plt.title('Alias method distribution')
        plt.legend()
    return mean_alias


alias_distribution = alias(P,x, sample_size)


## Part 3 compare the three generation methods using appropriate measures and discuss the results
Possible criteria
- computational efficiency
- Ease of implementation 
- memory requirements
- accuracy of the generated distribution

In [ ]:
# time complexity:
from time import time

np.random.seed(67)
x = np.array([1,2,3,4,5,6])
P = np.array([7/48, 5/48, 1/8, 1/16, 1/4, 5/16])

S = np.arange(1_000, 100_000, 1000)

crude_time =[]
reject_time = []
alias_time = []

crude_accuracy=[]
reject_accuracy = []
alias_accuracy = []

import numpy as np
from scipy.stats import chisquare


def get_chi_sq_p_value_from_probs(estimated_probs, P, sample_size):
    # 1. Convert estimated probabilities back to frequency counts
    # This reverses the division by s
    # Inside the function
    estimated_probs = estimated_probs / np.sum(estimated_probs)
    observed_counts = estimated_probs * sample_size
    
    # 2. Expected counts (True P * s)
    expected_counts = P * sample_size
    
    # 3. Perform Chi-Squared test
    # Note: Use a small epsilon to avoid division by zero if an outcome has 0 prob
    stat, p_value = chisquare(f_obs=observed_counts, f_exp=expected_counts)
    return p_value



for s in S:
    t = time()
    crude_dist_ = crude_dist(P,x,s, plot_=False)
    crude_time.append(time()-t)

    t = time()
    rejection_dist = rejection_method(P,x,sample_size, plot_=False)
    reject_time.append(time()-t)
    t = time()
    alias_dist = alias(P,x,sample_size,plot_=False)
    alias_time.append(time()-t)

    crude_p_val = get_chi_sq_p_value_from_probs(crude_dist_, P, s)
    crude_accuracy.append(crude_p_val)
    reject_p_val = get_chi_sq_p_value_from_probs(rejection_dist, P, s)
    reject_accuracy.append(reject_p_val)
    alias_p_val = get_chi_sq_p_value_from_probs(alias_dist, P, s)
    alias_accuracy.append(alias_p_val)


    

In [ ]:

plt.plot(S,crude_time, label='crude')
plt.plot(S,reject_time, label='reject method')
plt.plot(S,alias_time, label='alias')
plt.title('Comparisons of time usage')
plt.legend()
plt.xlabel('sample size')

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [crude_accuracy, reject_accuracy, alias_accuracy],
    labels=['crude', 'reject method', 'alias'],
    patch_artist=True
)
plt.title('Comparisons of Accuracy in Chi test')
plt.xlabel('Method')
plt.ylabel('Chi-square p-value')
plt.tight_layout()


# Exercise 3

In [ ]:
# Part 1 
#1. Generate simulated values from the following distributions
#(a) Exponential distribution
#(b) Normal distribution (at least with standard Box-Mueller)
#(c) Pareto distribution, with β = 1 and experiment with
#diﬀerent values of k values: k = 2.05, k = 2.5, k = 3 and k = 4.
sample_size = 10_000
U = np.random.uniform(0,1,sample_size)

lam = 1
Y = -np.log(U)/lam
n, count = np.unique(Y, return_counts=True)
theoretical_exponential = lam*np.exp(-lam*n)
plt.hist(Y, color='r', bins=np.arange(1,np.max(Y))-1/2, density=True, width=0.5)
plt.plot(n,theoretical_exponential, color='b')

In [ ]:
# box-muller normal distribution
import numpy as np
import matplotlib.pyplot as plt


np.random.seed(67)
sample_size = 10_000

U1 = np.random.uniform(0, 1, sample_size)
U2 = np.random.uniform(0, 1, sample_size)

X = np.sqrt(-2 * np.log(U1)) * np.cos(2 * np.pi * U2)


plt.hist(X, bins=100, density=True, color='r', alpha=0.6, label='Generated Data')



x_pdf = np.linspace(X.min(), X.max(), 400)
pdf = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x_pdf**2)

plt.plot(x_pdf, pdf, c='b', lw=2, label='Theoretical PDF')
plt.legend()
plt.show()

In [ ]:
# Pareto distribution 

beta = 1
K = np.array([2.05, 2.5, 3, 4])
U = np.random.uniform(0, 1, sample_size)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))

for i, k in enumerate(K):
    Y = beta * U**(-1 / k)
    ax[i].hist(Y, bins=100, density=True, color='r', alpha=0.6, label='Generated Data')

    x = np.linspace(beta, np.percentile(Y, 99), 400)
    Pareto_theoretical = (k * beta**k) / (x**(k + 1))
    ax[i].plot(x, Pareto_theoretical, c='b', lw=2, label='Theoretical Pareto')
    ax[i].set_title(f'k = {k}')
    ax[i].legend()

plt.tight_layout()


In [ ]:
# 2. Pareto dsitribution 
import pandas as pd
means_sim = []
vars_sim = []

means_theo = []
vars_theo = []



for k in K:
    mu_E= beta * k/(k-1)
    var_E = beta**2 * k/((k-1)**2*(k-2))

    Y = beta * U**(-1 / k)
    means_sim.append(np.mean(Y))
    vars_sim.append(np.var(Y))

    means_theo.append(mu_E)
    vars_theo.append(var_E)

data = {
    "k": K,
    "Mean (Sim)": means_sim,
    "Mean (Theo)": means_theo,
    "Var (Sim)": vars_sim,
    "Var (Theo)": vars_theo
}


df = pd.DataFrame(data)

print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    

In [ ]:
# Part 3 compare confidence intervals of normal distributions

import numpy as np
import scipy.stats as stats

# Parameters
n = 10
iterations = 100
true_mu = 0
true_var = 1
confidence = 0.95

mean_hits = 0
var_hits = 0

for _ in range(iterations):
    # Generate 10 observations
    sample = np.random.normal(true_mu, np.sqrt(true_var), n)
    
    sample_mean = np.mean(sample)
    sample_var = np.var(sample, ddof=1)
    
    # 1. 95% CI for Mean (using t-distribution)
    # interval = mean +/- t_crit * (std_err)
    t_crit = stats.t.ppf((1 + confidence) / 2, df=n-1)
    margin_mean = t_crit * np.sqrt(sample_var / n)
    if (sample_mean - margin_mean) <= true_mu <= (sample_mean + margin_mean):
        mean_hits += 1
        
    # 2. 95% CI for Variance (using chi-squared distribution)
    # interval = [(n-1)*s^2 / chi2_upper, (n-1)*s^2 / chi2_lower]
    chi2_lower = stats.chi2.ppf((1 + confidence) / 2, df=n-1)
    chi2_upper = stats.chi2.ppf((1 - confidence) / 2, df=n-1)
    
    var_lower = (n - 1) * sample_var / chi2_lower
    var_upper = (n - 1) * sample_var / chi2_upper
    
    if var_lower <= true_var <= var_upper:
        var_hits += 1

print(f"Mean CIs containing true mean: {mean_hits}/{iterations}")
print(f"Variance CIs containing true variance: {var_hits}/{iterations}")